# 05 · Ablation over the state vector  *(branch `exp/05-rl-ablation`)* — the core result

Brief §3.4 (a). **Same agent (PPO), same hyperparameters**, four states, 5 seeds each:

| | state | dim | answers |
|---|---|---|---|
| A | `[position, time-in-position] + 8 factors` | 10 | control |
| B | A + sentiment | 11 | RQ1 |
| C | A + risk | 11 | isolates risk |
| D | A + both | 12 | RQ3 (vs B) |

LLM signals are the **previous completed day's** aggregate (causal). `LEAK_CHECK` additionally runs D with the
dataset's same-day alignment — the difference measures how much the look-ahead would have inflated the result.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/05-rl-ablation"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "05_rl_ablation")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run_seeds
from rltrade import results as R
SEEDS = [0, 1, 2, 3, 4]
AGENT, AGENT_HP = "ppo", {}          # notebook PPO defaults - identical for A-D
TOTAL_STEPS, EVAL_EVERY = 2_000_000, 100_000
LEAK_CHECK = True
F = paths.factor_dir(L)

In [ ]:
for state in ("A", "B", "C", "D"):
    run_seeds(RunSpec("05_ablation", AGENT, env=EnvConfig(state=state), agent_hp=AGENT_HP,
                      total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY), SEEDS, L.prepared, F, L.runs)
run_seeds(RunSpec("05_ablation", "buy_and_hold", env=EnvConfig(state="D")), [0], L.prepared, F, L.runs)
if LEAK_CHECK:
    run_seeds(RunSpec("05_ablation", AGENT, env=EnvConfig(state="D", signal_source="sameday"), agent_hp=AGENT_HP,
                      total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY), SEEDS, L.prepared, F, L.runs)

## Results

In [ ]:
df = R.load_results(L.runs); df = df[df.experiment == "05_ablation"]
ab = df[(df.agent == AGENT) & (df.signal_source == "lagged")]
sm = R.summarise(ab)
table = R.markdown_table(sm, group=("state",))
print(table)
R.plot_ablation(sm, L.report / "ablation_bars.png"); plt.show()

def compare(a, b, metric="test_sharpe"):
    x, y = ab[ab.state == a][metric].values, ab[ab.state == b][metric].values
    t = stats.ttest_ind(x, y, equal_var=False)
    return {"pair": f"{a} - {b}", "diff_mean": x.mean() - y.mean(), "welch_t": t.statistic, "p": t.pvalue, "n": (len(x), len(y))}
tests = pd.DataFrame([compare(*p) for p in (("B", "A"), ("C", "A"), ("D", "A"), ("D", "B"))])
print(tests.round(4))       # n = 5 per group: treat p-values as indicative only
(L.report / "ablation.md").write_text("# Ablation A-D (test, net of costs)\n\n" + table + "\n\n" + R.df_markdown(tests, 4) + "\n", encoding="utf-8")

In [ ]:
groups = {f"PPO {s}": ab[ab.state == s] for s in "ABCD"}
groups["Buy & hold"] = df[df.agent == "buy_and_hold"]
R.plot_curves(df, L.runs, groups, path=L.report / "ablation_equity.png"); plt.show()
R.plot_curves(df, L.runs, groups, kind="drawdown", path=L.report / "ablation_drawdown.png"); plt.show()
if LEAK_CHECK:
    lk = df[(df.agent == AGENT) & (df.state == "D")]
    print(R.markdown_table(R.summarise(lk), group=("state", "signal_source")))